# 项目二：联网查询机器人 — Tool Use（工具调用）

在项目一中，我们学会了调用 DeepSeek API 让大模型"思考"和"回答"。但你会发现一个问题：

> **大模型本身不能上网搜索、不能查数据库、不能操作文件——它只能"说"，不能"做"。**

本项目将解决这个问题。你将学习：

1. 理解 **Tool Use（工具调用）** 的核心思想
2. 掌握 DeepSeek API 的 **Function Calling** 机制
3. 学会定义工具描述（`tools`）并解析模型的调用指令
4. 构建一个能"联网查天气"和"查数据库"的智能体

---

## 一、什么是 Tool Use？

### 1.1 一个生活中的类比

想象你是一个非常聪明但**被关在房间里**的人：
- 你可以思考、推理、写文章 ✅
- 但你不能出门看天气、不能打电话问朋友、不能打开电脑查资料 ❌

现在，房间开了一个小窗口，你可以递纸条出去，外面有一个助手会帮你执行操作并把结果递回来。

这就是 **Tool Use** 的本质：

```
用户提问 → 大模型思考 → 决定调用某个工具 → 系统执行工具 → 把结果返回给大模型 → 大模型组织最终回答
```

### 1.2 为什么需要 Tool Use？

| 大模型的局限 | Tool Use 的解决方式 |
|-------------|-------------------|
| 知识有截止日期，不知道最新信息 | 调用搜索引擎获取实时数据 |
| 不能精确计算（如 1234 × 5678） | 调用计算器工具 |
| 不能访问私有数据库 | 调用数据库查询工具 |
| 不能执行代码 | 调用代码执行沙箱 |
| 容易产生幻觉（编造事实） | 调用权威数据源获取真实信息 |

### 1.3 DeepSeek 的 Function Calling 机制

DeepSeek（和 OpenAI）提供了一种标准化的工具调用方式，核心流程如下：

```
┌──────────────────────────────────────────────────────────┐
│  第 1 步：我们告诉模型有哪些工具可用（tools 参数）          │
│  第 2 步：用户提问                                        │
│  第 3 步：模型判断是否需要调用工具                           │
│     ├── 不需要 → 直接返回文字回答                           │
│     └── 需要   → 返回工具名称 + 参数（JSON 格式）           │
│  第 4 步：我们的代码执行工具，拿到结果                       │
│  第 5 步：把工具结果再次发给模型                            │
│  第 6 步：模型根据工具结果，生成最终的自然语言回答            │
└──────────────────────────────────────────────────────────┘
```

---

## 二、基础准备

### 2.1 安装依赖 & 导入库

本项目除了 `openai` 之外，还需要 `requests` 来调用真实的天气 API：

In [ ]:
# Install dependencies for JupyterLite/Pyodide.
# Shell-style pip commands are not available in this browser runtime.
import micropip
await micropip.install(["openai", "requests"])
print("Dependencies installed: openai, requests")


In [ ]:
import os
import json
import requests
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY", "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"),
    base_url="https://api.deepseek.com",
)

print("客户端初始化完成！")

### 2.2 如何定义工具？

在 API 调用中，我们通过 `tools` 参数告诉模型"你有哪些工具可以用"。

每个工具需要用 **JSON Schema** 格式描述，包含三部分：

| 字段 | 含义 | 说明 |
|------|------|------|
| `name` | 工具名称 | 模型会根据这个名字来指定调用哪个工具 |
| `description` | 工具描述 | 告诉模型这个工具是干什么的，什么时候该用 |
| `parameters` | 参数定义 | 告诉模型调用这个工具需要传什么参数 |

来看一个具体的例子——定义一个"查天气"的工具：

```python
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",                  # 工具名称
            "description": "查询指定城市的天气情况",    # 描述：模型据此判断何时调用
            "parameters": {                          # 参数定义
                "type": "object",
                "properties": {                      # 有哪些参数
                    "city": {
                        "type": "string",
                        "description": "城市名称，如'北京'、'上海'"
                    }
                },
                "required": ["city"]                 # 哪些参数是必填的
            }
        }
    }
]
```

### 2.3 两种新的消息角色：assistant（带 tool_calls）和 tool

在项目一中，我们只用了 `system` 和 `user` 两种角色。但在 Tool Use 场景下，还需要用到两种**新的角色**来完成工具调用的完整流程。

#### 角色一：assistant（带 tool_calls 字段）

当模型判断需要调用工具时，它返回的 `assistant` 消息不再是普通的文字回复，而是包含一个 `tool_calls` 字段：

```python
# 普通回复（项目一中的情况）
{"role": "assistant", "content": "你好！"}

# 带 tool_calls 的回复（项目二的新情况）
{
    "role": "assistant",
    "content": None,              # 注意：此时 content 为空！
    "tool_calls": [               # 工具调用列表（可以有多个）
        {
            "id": "call_abc123",  # 工具调用的唯一标识符
            "type": "function",   # 固定值，表示函数调用
            "function": {
                "name": "get_weather",                    # 要调用的函数名
                "arguments": "{\"city\": \"Beijing\"}"     # 参数（JSON 字符串）
            }
        }
    ]
}
```

**关键字段说明：**

| 字段 | 类型 | 说明 |
|------|------|------|
| `content` | `None` | 当模型决定调用工具时，content 为空 |
| `tool_calls` | `list` | 工具调用列表，模型可能一次调用多个工具 |
| `tool_calls[].id` | `string` | 每次调用的唯一 ID，后续需要用这个 ID 关联结果 |
| `tool_calls[].type` | `string` | 固定为 `"function"` |
| `tool_calls[].function.name` | `string` | 要调用的函数名称 |
| `tool_calls[].function.arguments` | `string` | 函数参数，**是 JSON 格式的字符串**，需要 `json.loads()` 解析 |

---

#### 角色二：tool（工具执行结果）

当我们执行完本地函数后，需要把结果通过 `tool` 角色的消息返回给模型：

```python
{
    "role": "tool",                    # 固定为 "tool"
    "tool_call_id": "call_abc123",     # 对应 assistant 消息中 tool_call 的 id
    "content": "Beijing当前天气：晴，气温 28°C"  # 工具执行的结果（字符串）
}
```

**关键字段说明：**

| 字段 | 类型 | 说明 |
|------|------|------|
| `role` | `"tool"` | 固定值，表示这是工具执行结果 |
| `tool_call_id` | `string` | **必须**与 assistant 消息中某个 tool_call 的 id 完全匹配 |
| `content` | `string` | 工具执行的结果，会作为上下文传给模型 |

**⚠️ 重要规则：**
- 每个 `tool_call_id` 都**必须**有一条对应的 `tool` 消息
- 如果模型返回了 3 个 tool_calls，就必须有 3 条 tool 消息
- 否则会报错：`"insufficient tool messages following tool_calls message"`

---

#### 完整的消息流转示例

下面展示一次完整的工具调用过程中，messages 列表是如何变化的：

```python
# 第 1 次 API 调用：用户提问
messages = [
    {"role": "user", "content": "北京今天天气怎么样？"}
]
# → 模型返回：assistant 消息，包含 tool_calls

# 第 2 次 API 调用：带上完整的对话历史
messages = [
    {"role": "user", "content": "北京今天天气怎么样？"},
    {"role": "assistant", "content": None, "tool_calls": [...]},  # ← 模型的调用指令
    {"role": "tool", "tool_call_id": "call_abc123", "content": "晴天，28°C"},  # ← 工具结果
]
# → 模型根据工具结果，生成最终的自然语言回答
```

---

## 三、示例一：天气查询机器人 🌤️

让我们从零开始，完整实现一个能查天气的智能体。

### 第一步：定义本地工具函数

这是真正执行功能的 Python 函数。我们使用免费的 [wttr.in](https://wttr.in) 天气 API 获取**真实天气数据**，无需注册、无需 API Key。

> `wttr.in` 是一个开源天气服务，返回 JSON 格式的天气数据，非常适合学习和原型开发。

In [ ]:
# ===== 本地工具函数（调用真实天气 API）=====

def get_weather(city: str) -> str:
    """查询指定城市的实时天气（使用 wttr.in 免费 API）"""
    try:
        # wttr.in 返回 JSON 格式的天气数据，lang=zh 表示返回中文描述
        resp = requests.get(
            f"https://wttr.in/{city}?format=j1&lang=zh",
            timeout=10
        )
        data = resp.json()
        
        # 从返回的 JSON 中提取关键天气信息
        current = data["current_condition"][0]
        temp = current["temp_C"]            # 气温（摄氏度）
        humidity = current["humidity"]       # 湿度
        desc = current["lang_zh"][0]["value"]  # 中文天气描述
        feels_like = current["FeelsLikeC"]   # 体感温度
        wind_speed = current["windspeedKmph"]  # 风速
        
        return f"{city}当前天气：{desc}，气温 {temp}°C，体感 {feels_like}°C，湿度 {humidity}%，风速 {wind_speed}km/h"
    except Exception as e:
        return f"获取 {city} 天气失败：{e}"


# 测试工具函数（将返回真实天气数据！）
print(get_weather("Kaifeng"))
print(get_weather("Zhengzhou"))

### 第二步：定义工具描述（告诉模型有哪些工具可用）

In [ ]:
# ===== 工具描述（JSON Schema）=====
# 注意：这不是执行函数，而是给模型看的"说明书"

weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "根据城市名称查询当前的实时天气情况，包括温度、天气状况、湿度和风速",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "要查询天气的城市英文名（拼音），例如：Beijing、Shanghai、Guangzhou"
                }
            },
            "required": ["city"]
        }
    }
}

print(json.dumps(weather_tool, ensure_ascii=False, indent=2))

### 第三步：构建智能体——完整的工具调用流程

这是最关键的部分。我们一步步来看模型是如何"决定"调用工具的：

In [ ]:
def weather_agent(user_query):
    """天气查询智能体：支持多轮工具调用"""
    
    print(f"👤 用户提问：{user_query}\n")
    
    # 初始化消息列表
    messages = [{"role": "user", "content": user_query}]
    
    # 使用 while 循环支持多轮工具调用
    round_num = 0
    while True:
        round_num += 1
        print(f"--- 第 {round_num} 轮 API 调用 ---")
        
        # 发送请求
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            tools=[weather_tool],
        )
        
        assistant_message = response.choices[0].message
        
        # 检查模型是否要调用工具
        if not assistant_message.tool_calls:
            # 模型返回最终回答，没有工具调用
            print(f" 最终回答：{assistant_message.content}")
            return assistant_message.content
        
        # 模型决定调用工具！遍历所有工具调用
        print(f" 模型决定调用 {len(assistant_message.tool_calls)} 个工具")
        
        # 构建 assistant 消息（SDK 对象 → 字典格式）
        tool_calls_dict = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            }
            for tc in assistant_message.tool_calls
        ]
        
        # 添加 assistant 消息到历史记录
        # 注意：content 必须是空字符串 ""，不能是 None
        messages.append({
            "role": "assistant",
            "content": "",
            "tool_calls": tool_calls_dict
        })
        
        # 执行每个工具，并添加对应的 tool 消息
        for tc in assistant_message.tool_calls:
            function_name = tc.function.name
            arguments = json.loads(tc.function.arguments)
            
            print(f"  调用工具：{function_name}({arguments})")
            
            # 执行本地工具函数
            if function_name == "get_weather":
                tool_result = get_weather(**arguments)
            else:
                tool_result = f"未知工具：{function_name}"
            
            print(f"  执行结果：{tool_result}")
            
            # 添加 tool 消息（每个 tool_call 都必须有一条对应的 tool 消息）
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": tool_result
            })
        
        print()  # 空行
        
        # 循环继续，让模型根据工具结果生成回答（或继续调用工具）
        if round_num >= 5:
            print(" 达到最大轮次限制，停止调用")
            break


# ===== 测试（将获取真实天气数据！）=====
print("=" * 50)
weather_agent("北京今天天气怎么样？")
print("\n" + "=" * 50)
weather_agent("上海和广州哪个更热？")
print("\n" + "=" * 50)
weather_agent("你好，请做一下自我介绍")  # 这个不需要调用工具

### 流程图回顾

```
用户："北京今天天气怎么样？"
        │
        ▼
┌─────────────────────┐
│ 发送给模型 + tools   │ ← 告诉模型你有 get_weather 工具
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 模型返回 tool_calls  │ ← 模型说："我要调用 get_weather(city='Beijing')"
└─────────┬───────────┘
          │
          ▼
┌─────────────────────┐
│ 执行本地 get_weather │ ← 调用 wttr.in API，得到真实天气数据
└─────────┬───────────┘
          │
          ▼
┌─────────────────────────────┐
│ 把工具结果再次发给模型        │ ← role: "tool", content: "Beijing当前天气：晴..."
└─────────┬───────────────────┘
          │
          ▼
┌─────────────────────────────┐
│ 模型生成自然语言最终回答      │ ← "北京今天天气晴朗，气温28°C..."
└─────────────────────────────┘
```

---

## 四、示例二：多工具协作——天气 + 日期计算器

在真实场景中，智能体通常拥有**多个工具**，需要自己判断该用哪个。

下面我们注册两个工具，让模型自己决定调用哪个：

| 工具 | 功能 | 触发场景 |
|------|------|----------|
| `get_weather` | 查天气 | 用户问天气相关 |
| `calculate_days` | 计算两个日期之间相差几天 | 用户问日期计算相关 |

### 第一步：定义两个本地工具函数

In [ ]:
from datetime import datetime

def get_weather(city: str) -> str:
    """查询指定城市的实时天气（使用 wttr.in 免费 API）"""
    try:
        resp = requests.get(
            f"https://wttr.in/{city}?format=j1&lang=zh",
            timeout=10
        )
        data = resp.json()
        current = data["current_condition"][0]
        temp = current["temp_C"]
        humidity = current["humidity"]
        desc = current["lang_zh"][0]["value"]
        return f"{city}当前天气：{desc}，气温 {temp}°C，湿度 {humidity}%"
    except Exception as e:
        return f"获取 {city} 天气失败：{e}"


def calculate_days(date1: str, date2: str) -> str:
    """计算两个日期之间相差多少天
    参数:
        date1: 起始日期，格式 YYYY-MM-DD
        date2: 结束日期，格式 YYYY-MM-DD
    """
    d1 = datetime.strptime(date1, "%Y-%m-%d")
    d2 = datetime.strptime(date2, "%Y-%m-%d")
    diff = abs((d2 - d1).days)
    return f"{date1} 到 {date2} 相差 {diff} 天"


# 测试
print(get_weather("Shanghai"))
print(calculate_days("2025-01-01", "2025-10-01"))

### 第二步：定义两个工具的描述

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "根据城市名称查询当前实时天气情况，包括温度、天气状况和湿度",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "要查询天气的城市英文名（拼音），例如：Beijing、Shanghai"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_days",
            "description": "计算两个日期之间相差多少天",
            "parameters": {
                "type": "object",
                "properties": {
                    "date1": {
                        "type": "string",
                        "description": "起始日期，格式为 YYYY-MM-DD"
                    },
                    "date2": {
                        "type": "string",
                        "description": "结束日期，格式为 YYYY-MM-DD"
                    }
                },
                "required": ["date1", "date2"]
            }
        }
    }
]

print(f"已注册 {len(tools)} 个工具：")
for t in tools:
    print(f"  - {t['function']['name']}: {t['function']['description']}")

### 第三步：构建多工具智能体

注意观察：当有多个工具时，我们需要一个**路由机制**——根据模型返回的工具名，调用不同的本地函数。

In [ ]:
# ===== 工具注册表：把工具名映射到实际的 Python 函数 =====
TOOL_REGISTRY = {
    "get_weather": get_weather,
    "calculate_days": calculate_days,
}


def smart_agent(user_query):
    """多工具智能体：自动选择并调用合适的工具"""
    
    print(f"👤 用户提问：{user_query}\n")
    
    # 第 1 步：发送请求，带上所有工具
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": user_query}],
        tools=tools,
    )
    
    assistant_message = response.choices[0].message
    
    # 第 2 步：检查是否有工具调用
    if assistant_message.tool_calls:
        # 支持一次调用多个工具（并行工具调用）
        tool_results = []
        
        for tool_call in assistant_message.tool_calls:
            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            
            print(f"🤖 调用工具：{function_name}({arguments})")
            
            # ===== 路由：根据工具名找到对应的函数并执行 =====
            if function_name in TOOL_REGISTRY:
                result = TOOL_REGISTRY[function_name](**arguments)
            else:
                result = f"错误：未知工具 {function_name}"
            
            print(f"🔧 执行结果：{result}\n")
            tool_results.append((tool_call.id, result))
        
        # 第 3 步：把所有工具结果返回给模型
        messages = [{"role": "user", "content": user_query}]
        tool_calls_dict = [
            {
                "id": tc.id,
                "type": "function",
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            }
            for tc in assistant_message.tool_calls
        ]
        messages.append({
            "role": "assistant",
            "content": "",
            "tool_calls": tool_calls_dict
        })
        for tool_id, result in tool_results:
            messages.append({
                "role": "tool",
                "tool_call_id": tool_id,
                "content": result
            })
        
        final_response = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            tools=tools,
        )
        
        final_answer = final_response.choices[0].message.content
        print(f"✅ 最终回答：{final_answer}")
        return final_answer
    
    else:
        print(f"🤖 直接回答：{assistant_message.content}")
        return assistant_message.content


# ===== 测试：观察模型如何选择不同的工具 =====
print("=" * 50)
smart_agent("上海今天天气如何？")

print("\n" + "=" * 50)
smart_agent("2025年1月1日到2025年国庆节相差多少天？")

print("\n" + "=" * 50)
smart_agent("你是谁？能做什么？")  # 不需要工具

### 关键收获

对比示例一和示例二，核心区别在于：

| | 示例一（单工具） | 示例二（多工具） |
|---|---|---|
| `tools` 参数 | 只有 1 个工具 | 有多个工具 |
| 路由逻辑 | `if function_name == "get_weather"` | 使用 `TOOL_REGISTRY` 字典映射 |
| 模型判断 | 只有天气相关才调用 | 模型自己决定用哪个工具 |
| 扩展性 | 加工具要改 if-else | 只需在注册表中加一行 |

---

## 五、课后练习

### 练习 1（基础）：单位换算工具

**目标：** 构建一个能进行单位换算的智能体。

**要求：**
1. 实现一个 `convert_unit` 函数，支持以下换算：
   - 温度：摄氏度 ↔ 华氏度
   - 长度：千米 ↔ 英里
   - 重量：千克 ↔ 磅
2. 编写工具的 JSON Schema 描述
3. 构建智能体，能根据用户输入自动调用换算工具

**提示：**
- 函数参数可以设计为 `value`（数值）、`from_unit`（原单位）、`to_unit`（目标单位）
- 摄氏度转华氏度：`F = C × 9/5 + 32`
- 1 千米 ≈ 0.621371 英里
- 1 千克 ≈ 2.20462 磅

**测试用例：**
- "帮我把 36.5 摄氏度转换成华氏度"
- "100 千米等于多少英里？"
- "今天天气怎么样？"（应该不调用工具）

In [ ]:
# ===== 练习 1：单位换算工具 =====

# 第一步：实现本地工具函数
def convert_unit(value: float, from_unit: str, to_unit: str) -> str:
    """单位换算函数"""
    # TODO: 实现以下换算逻辑
    # 温度：摄氏度 ↔ 华氏度  (F = C × 9/5 + 32)
    # 长度：千米 ↔ 英里      (1 千米 ≈ 0.621371 英里)
    # 重量：千克 ↔ 磅        (1 千克 ≈ 2.20462 磅)
    # 不支持的换算返回错误提示
    pass


# 第二步：编写工具的描述（JSON Schema）
unit_tool = {
    "type": "function",
    "function": {
        "name": "convert_unit",
        "description": "TODO: 描述这个工具的功能",
        "parameters": {
            # TODO: 补全参数定义（value, from_unit, to_unit）
        }
    }
}


# 第三步：构建智能体
def unit_agent(user_query):
    """单位换算智能体"""
    # TODO: 参考示例一，补全完整的工具调用流程
    # 1. 调用 API（带 tools 参数）
    # 2. 检查是否有 tool_calls
    # 3. 如果有，解析并执行 convert_unit
    # 4. 把结果返回给模型，获取最终回答
    pass


# ===== 测试 =====
print("测试 1：温度换算")
unit_agent("帮我把 36.5 摄氏度转换成华氏度")

print("\n测试 2：长度换算")
unit_agent("100 千米等于多少英里？")

print("\n测试 3：不触发工具")
unit_agent("今天天气怎么样？")


### 练习 2（进阶）：个人信息查询系统

**目标：** 构建一个拥有**多个工具**的"企业员工信息查询系统"，模拟真实业务场景。

**要求：**
1. 实现以下 3 个工具函数：
   - `search_employee(name)` — 根据姓名查询员工信息（部门、职位、工号）
   - `get_department_info(department)` — 查询部门信息（负责人、人数、楼层）
   - `send_message(name, message)` — 给某员工发送消息（返回发送状态）
2. 编写 3 个工具的 JSON Schema 描述
3. 使用**工具注册表**（字典映射）构建智能体
4. 智能体需要能根据用户意图选择合适的工具

**模拟数据：**
```python
EMPLOYEE_DB = {
    "张三": {"department": "技术部", "position": "高级工程师", "id": "EMP001"},
    "李四": {"department": "产品部", "position": "产品经理", "id": "EMP002"},
    "王五": {"department": "技术部", "position": "前端开发", "id": "EMP003"},
}

DEPARTMENT_DB = {
    "技术部": {"leader": "赵总", "headcount": 50, "floor": "8楼"},
    "产品部": {"leader": "钱总", "headcount": 20, "floor": "7楼"},
}
```

**测试用例：**
- "张三是什么职位？"
- "技术部在几楼？负责人是谁？"
- "帮我给李四发一条消息：明天开会"
- "技术部有哪些人？"（思考：模型会怎么处理这个问题？）

In [ ]:
# ===== 练习 2：个人信息查询系统 =====

# 模拟数据库
EMPLOYEE_DB = {
    "张三": {"department": "技术部", "position": "高级工程师", "id": "EMP001"},
    "李四": {"department": "产品部", "position": "产品经理", "id": "EMP002"},
    "王五": {"department": "技术部", "position": "前端开发", "id": "EMP003"},
}

DEPARTMENT_DB = {
    "技术部": {"leader": "赵总", "headcount": 50, "floor": "8楼"},
    "产品部": {"leader": "钱总", "headcount": 20, "floor": "7楼"},
}


# 第一步：实现 3 个工具函数
def search_employee(name: str) -> str:
    """根据姓名查询员工信息"""
    # TODO: 从 EMPLOYEE_DB 中查找，返回格式化的员工信息
    # 找不到时返回友好提示
    pass


def get_department_info(department: str) -> str:
    """查询部门信息"""
    # TODO: 从 DEPARTMENT_DB 中查找，返回格式化的部门信息
    pass


def send_message(name: str, message: str) -> str:
    """给员工发送消息"""
    # TODO: 检查员工是否存在，返回发送状态
    pass


# 第二步：编写 3 个工具的描述（JSON Schema）
employee_tools = [
    # TODO: 参考示例二，为 3 个工具分别编写 JSON Schema
    # 提示：注意每个工具的 parameters 和 required 字段
]


# 第三步：构建工具注册表
TOOL_REGISTRY = {
    # TODO: 把工具名映射到函数
}


# 第四步：构建智能体
def company_agent(user_query):
    """企业信息查询智能体"""
    # TODO: 参考示例二的 smart_agent，补全完整流程
    pass


# ===== 测试 =====
print("测试 1：查询员工")
company_agent("张三是什么职位？")

print("\n测试 2：查询部门")
company_agent("技术部在几楼？")

print("\n测试 3：发送消息")
company_agent("帮我给李四发一条消息：明天下午三点开会")

print("\n测试 4：不触发工具")
company_agent("你好")

---
## 小结

恭喜你完成了第二个项目！回顾一下核心知识点：

| 知识点 | 要点 |
|--------|------|
| Tool Use 的本质 | 大模型负责"决策"，外部工具负责"执行" |
| `tools` 参数 | 用 JSON Schema 描述工具的名称、功能、参数 |
| `tool_calls` | 模型返回的工具调用指令，包含函数名和参数 |
| `role: "tool"` | 把工具执行结果返回给模型的消息角色 |
| 工具注册表 | 用字典映射工具名到函数，比 if-else 更易扩展 |
| 多工具协作 | 模型根据用户意图自动选择合适的工具 |

**核心流程口诀：** 定义工具 → 告诉模型 → 模型决策 → 执行工具 → 结果回传 → 模型总结

**下一步预告：** 在[项目三](./project3_memory.ipynb)中，我们将学习如何让智能体拥有**记忆**，实现多轮对话上下文管理！